# RAG Pipeline - Data Ingesion to Vector DB

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

### 1. Document Structure

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


def process_all_pdfs(pdf_directory):
    """Load all PDF files from a directory and its subdirectories."""

    all_documents = []

    # Convert folder path string into a Path object
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.rglob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file}")

        try:
            # Load PDF
            documents = PyPDFLoader(str(pdf_file)).load()

            # Add metadata to every page
            for doc in documents:
                doc.metadata.update({
                    "source": str(pdf_file),
                    "file_type": "pdf"
                })

            # Add documents to the main list
            all_documents.extend(documents)

            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents

In [7]:
# Process all PDFs inside the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 7 PDF files to process

Processing: ..\data\pdf\AWS services explained.pdf
  ✓ Loaded 18 pages

Processing: ..\data\pdf\CSS Notes.pdf
  ✓ Loaded 72 pages

Processing: ..\data\pdf\Customer relationship management.pdf
  ✓ Loaded 21 pages

Processing: ..\data\pdf\DSA all topics.pdf


Ignoring wrong pointing object 11 0 (offset 0)


  ✓ Loaded 13 pages

Processing: ..\data\pdf\git-cheat-sheet-education.pdf
  ✓ Loaded 2 pages

Processing: ..\data\pdf\Java DBMS interview questions.pdf
  ✓ Loaded 56 pages

Processing: ..\data\pdf\React Node Express MERN FAQ.pdf
  ✓ Loaded 43 pages

Total documents loaded: 225


### 2. Splitter/CHUNK

In [8]:
def split_documnents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len, # len to count character we can haev custom fuction for this
        separators=["\n\n","\n"," ",""] # can add '.' bcz mr. or 6.2
    )
    
    all_chunks=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(all_chunks)} chunks")
    
    if all_chunks:
        print(f"\nExample chunk")
        print(f"Content: {all_chunks[1].page_content}")
        print(f"Metadata:{all_chunks[1].metadata}")
    return all_chunks

In [9]:
chunks = split_documnents(all_pdf_documents)
chunks

Split 225 documents into 317 chunks

Example chunk
Content: 2/18
8. Monitoring & Logging
CloudWatch, CloudTrail, X-Ray
9. Analytics & Streaming
Kinesis, EMR, Athena, Glue
10. Machine Learning & AI
SageMaker
11. Management, Provisioning & Migration
CloudFormation, AWS Config, Systems Manager, DMS
12. Serverless & Event-driven patterns
API Gateway, EventBridge, Step Functions
13. IoT, Mobile & Edge
IoT Core, Amplify, Greengrass
14. Typical architecture patterns (with flowcharts)
Classic web app (EC2 + RDS)
Serverless web app (API Gateway + Lambda)
Data pipeline (Kinesis/Glue/Redshift)
15. Quick cheat-sheet: When to choose which service
16. Study tips & further reading
1) Quick intro & AWS global infrastructure
What is AWS (brief): Amazon Web Services (AWS) is a cloud provider offering a
large set of on‑demand services for compute, storage, networking, databases,
analytics, machine learning, security, management tools and more. These services
are available in multiple geographic Regions a

[Document(metadata={'producer': 'Skia/PDF m140', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36', 'creationdate': '2025-09-27T13:31:01+00:00', 'title': 'AWS services explained', 'moddate': '2025-09-27T13:31:01+00:00', 'source': '..\\data\\pdf\\AWS services explained.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='1/18\nAWS services explained\nexplain each service provide aws in detail with some \nflowchart and how it works and there client of each one \ntake longer and make a good note to read and understand\nAWS Services — Detailed Notes,\nFlowcharts & How They Work\nTable of contents\n1. Quick intro & AWS global infra\n2. Compute\nEC2, Lambda, ECS, EKS, Elastic Beanstalk\n3. Storage\nS3, EBS, EFS, Glacier (S3 Glacier)\n4. Databases\nRDS, Aurora, DynamoDB, ElastiCache, Redshift\n5. Networking & Content Delivery\nVPC, Route\xa053, ELB (ALB/NLB), CloudFront\n6. Security

### 3. Embedder

In [10]:
import numpy as np
import chromadb
import hashlib
from chromadb.config import Settings
from typing import List, Dict, Tuple, Any
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-V2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
        
    def _load_model(self):
        try:
            print(f"Loading embedding model {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding Dimesion: {self.model.get_embedding_dimension}")
        except Exception as e:
            print(f"Error Model loading {self.model_name} : {e}")
            raise
        
    def genetate_embedding(self, text:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("Model Not Loaded")
        
        print(f"Generating embedding for {len(text)} texts.")
        embedding=self.model.encode(text,show_progress_bar=True)
        print(f"Generating embedding for {len(text)} texts.")
        return embedding

In [12]:
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model all-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1852.36it/s]


Model loaded successfully. Embedding Dimesion: <bound method SentenceTransformer.get_embedding_dimension of SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)>


### VectorStore

In [13]:
class VectorStore:
    def __init__(self, collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        
        os.makedirs(self.persist_directory,exist_ok=True)
        
        self.client=chromadb.PersistentClient(path=self.persist_directory)
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description":"PDF documents RAG vector"
            }
        )
        
        print(f"Vector Store initialized : {self.collection_name}")
        print(f"Existing Documents : {self.collection.count()}")
        
    def add_documents(self,chunk:List[Any],embedd:np.ndarray):
        
        if not chunk:
            print("No record to upload")
            return
        
        if len(chunk)!=len(embedd):
            raise ValueError("Chunk and Embedd size doesnt match")
        
        
        ids=[]
        metadatas=[]
        embeddings=[]
        documents=[]
        
        for chk,ebd in zip(chunk,embedd):
            
            source=chk.metadata.get("source","unknown")
            unique_string=f"{source}:{chk.page_content}"
            doc_id=hashlib.sha256(unique_string.encode('utf-8')).hexdigest()
            ids.append(doc_id)
            
            documents.append(chk.page_content)
            
            metadata=dict(chk.metadata)
            metadata['content_length']=len(chk.page_content)
            
            metadatas.append(metadata)
            
            
            embeddings.append(ebd.tolist())
            
            
        self.collection.upsert(
            ids=ids,
            documents=documents,
            metadatas=metadatas,
            embeddings=embeddings
        )
        
        print(f"Processed {len(chunk)}")
        print(f"Total chunks {self.collection.count()}")
        
    def similar_search(self,query_embedding:np.ndarray,n_results=5):
        
        result=self.collection.query(query_embeddings=[query_embedding.tolist()],n_results=n_results)
        return result
        

In [14]:
vectorstore=VectorStore()
vectorstore

Vector Store initialized : pdf_documents
Existing Documents : 317


In [15]:
texts=[doc.page_content for doc in chunks]

embedding=embedding_manager.genetate_embedding(texts)

vectorstore.add_documents(chunks,embedding)

Generating embedding for 317 texts.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches: 100%|██████████| 10/10 [00:14<00:00,  1.42s/it]


Generating embedding for 317 texts.
Processed 317
Total chunks 317


In [2]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)


def RagBot(query):
    query_embedding=embedding_manager.genetate_embedding([query])[0]
    retrive_info=vectorstore.similar_search(query_embedding=query_embedding,n_results=5)
    documents = retrive_info['documents'][0]
    
    prompt = f"""
    You are a helpful question-answering assistant.

    Answer the question using ONLY the context below.

    Rules:
    - Do not use outside knowledge.
    - Do not hallucinate.
    - If the context does not contain the answer, say:
    "I don't have enough information in the provided context."
    - Keep the answer concise and clear.

    Context:
    {documents}

    Question:
    {query}

    Answer:
    """

    response = llm.invoke(prompt)

    return response.content
    


In [3]:
RagBot("About aws and s3")

NameError: name 'embedding_manager' is not defined